# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
from pathlib import Path
import pandas as pd
import scipy.stats

df = pd.read_csv("ames_housing.csv")

df.describe()
print(df.shape)

(1460, 10)


---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [5]:
# 필수 1 코드를 작성하세요.
import pandas as pd
from scipy import stats

ALPHA = 0.05
df = pd.read_csv("ames_housing.csv")

# 1. 세 집단에서 표본 추출
groups = {}
for q in [5, 6, 7]:
    groups[q] = df.loc[df["OverallQual"] == q, "SalePrice"].sample(n=20, random_state=5)

# 2. 표본 수, 평균, 표준편차
print("[기초통계]")
for q, s in groups.items():
    print(f"OverallQual={q} : n={s.size}, 평균={s.mean():,.1f}, 표준편차={s.std():,.1f}")

# 3. 독립성 설명
print("\n[독립성]")
print("OverallQual은 한 주택에 하나의 등급만 부여되므로 5·6·7 집단에")
print("동일한 주택이 중복으로 들어갈 수 없다. 세 표본은 서로 겹치지 않는")
print("주택들로 구성되며 한 집단의 값이 다른 집단에 영향을 주지 않는다.")

# 4. Shapiro-Wilk 정규성 검정
print("\n[Shapiro-Wilk 정규성 검정]")
normal_ok = True
for q, s in groups.items():
    sw = stats.shapiro(s)
    ok = sw.pvalue > ALPHA
    normal_ok = normal_ok and ok
    print(f"OverallQual={q} : W={sw.statistic:.4f}, p={sw.pvalue:.4f}"
          f" → {'정규성 만족' if ok else '정규성 위배'}")

# 5. Levene 등분산 검정
lev = stats.levene(groups[5], groups[6], groups[7], center="median")
equal_var = lev.pvalue > ALPHA
print(f"\n[Levene] 통계량={lev.statistic:.4f}, p={lev.pvalue:.4f}"
      f" → {'등분산 만족' if equal_var else '등분산 위배'}")

# 6. 가설
print("\n[가설]")
print("H0 : 세 집단의 모집단 평균 판매가격은 모두 같다.")
print("H1 : 적어도 한 집단의 모집단 평균 판매가격은 다르다.")

# 7. 일원배치 ANOVA
res = stats.f_oneway(groups[5], groups[6], groups[7])

# 8. 결과 출력 및 판단
print("\n[일원배치 ANOVA]")
print(f"F = {res.statistic:.4f}")
print(f"p = {res.pvalue:.4f}")

if res.pvalue < ALPHA:
    print(f"→ p < {ALPHA} : 귀무가설을 기각한다.")
    print("   세 집단 중 적어도 한 집단의 평균 판매가격은 다르다고 볼 수 있다.")
else:
    print(f"→ p ≥ {ALPHA} : 귀무가설을 기각하지 못한다.")
    print("   세 집단의 평균에 차이가 있다고 볼 근거가 부족하다.")

# 9. ANOVA의 한계 설명
print("\n[ANOVA 결과의 한계]")
print("ANOVA는 '세 집단이 모두 같은가'만 판단하는 전체 검정(omnibus test)이다.")
print("귀무가설이 기각되어도 어느 쌍에서 차이가 났는지는 알려주지 않는다.")
print("구체적인 쌍을 확인하려면 Tukey HSD 같은 사후검정이 필요하다.")

[기초통계]
OverallQual=5 : n=20, 평균=130,605.0, 표준편차=24,937.1
OverallQual=6 : n=20, 평균=167,826.6, 표준편차=41,944.6
OverallQual=7 : n=20, 평균=217,593.6, 표준편차=48,298.4

[독립성]
OverallQual은 한 주택에 하나의 등급만 부여되므로 5·6·7 집단에
동일한 주택이 중복으로 들어갈 수 없다. 세 표본은 서로 겹치지 않는
주택들로 구성되며 한 집단의 값이 다른 집단에 영향을 주지 않는다.

[Shapiro-Wilk 정규성 검정]
OverallQual=5 : W=0.9710, p=0.7760 → 정규성 만족
OverallQual=6 : W=0.9527, p=0.4096 → 정규성 만족
OverallQual=7 : W=0.9259, p=0.1290 → 정규성 만족

[Levene] 통계량=2.6516, p=0.0792 → 등분산 만족

[가설]
H0 : 세 집단의 모집단 평균 판매가격은 모두 같다.
H1 : 적어도 한 집단의 모집단 평균 판매가격은 다르다.

[일원배치 ANOVA]
F = 24.2456
p = 0.0000
→ p < 0.05 : 귀무가설을 기각한다.
   세 집단 중 적어도 한 집단의 평균 판매가격은 다르다고 볼 수 있다.

[ANOVA 결과의 한계]
ANOVA는 '세 집단이 모두 같은가'만 판단하는 전체 검정(omnibus test)이다.
귀무가설이 기각되어도 어느 쌍에서 차이가 났는지는 알려주지 않는다.
구체적인 쌍을 확인하려면 Tukey HSD 같은 사후검정이 필요하다.


### 필수 1 답변 작성란

- **Q1.**
-> 검정을 반복할수록 전체 부서에서 한 번 이상 제 1종 오류가 발생확률이 0.05보다 커지는 다중 비교 문제가 발생한다.

- **Q2.**
-> 집단 간 평균 차이를 나타내는 집단 간 변동을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

- **Q3.**
-> 있습니다. 따라서 적어도 한 집단의 모집단 평균 판매가격은 다르다.

- **Q4.**
-> 알 수 없음, 구체적인 집단 쌍은 Tukey HSD와 같은 사후 검정으로 확인해야 한다.


---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [7]:
# 필수 2 코드를 작성하세요.
import pandas as pd
from itertools import combinations
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

ALPHA = 0.05
df = pd.read_csv("ames_housing.csv")

groups = {}
for q in [5, 6, 7]:
    groups[q] = df.loc[df["OverallQual"] == q, "SalePrice"].sample(n=20, random_state=5)

# 1. 세 집단을 하나의 데이터프레임으로 결합 (long format)
anova_df = pd.concat(
    [pd.DataFrame({"SalePrice": s.values, "group": q}) for q, s in groups.items()],
    ignore_index=True
)
print("[anova_df]")
print(anova_df.head())
print(f"전체 행 수 : {len(anova_df)}")

# 2. 전체 평균
grand_mean = anova_df["SalePrice"].mean()
print(f"\n전체 평균 = {grand_mean:,.2f}")

group_stats = anova_df.groupby("group")["SalePrice"].agg(["count", "mean"])
print("\n[집단별 평균]")
print(group_stats)

# 3. ANOVA 표 구성요소 계산
# 집단 간 제곱합 : 각 집단 평균이 전체 평균에서 얼마나 떨어져 있는지를 표본 수로 가중
ss_between = ((group_stats["mean"] - grand_mean) ** 2 * group_stats["count"]).sum()

# 집단 내 제곱합 : 각 관측치가 자기 집단 평균에서 떨어진 정도의 합
ss_within = sum(((s - s.mean()) ** 2).sum() for s in groups.values())

k = len(groups)                 # 집단 수
N = len(anova_df)               # 전체 표본 수
df_between = k - 1
df_within = N - k

ms_between = ss_between / df_between
ms_within = ss_within / df_within
f_manual = ms_between / ms_within

anova_table = pd.DataFrame({
    "제곱합(SS)": [ss_between, ss_within, ss_between + ss_within],
    "자유도(df)": [df_between, df_within, N - 1],
    "평균제곱(MS)": [ms_between, ms_within, None],
    "F": [f_manual, None, None],
}, index=["집단 간", "집단 내", "전체"])

print("\n[ANOVA 표]")
print(anova_table.to_string(float_format=lambda x: f"{x:,.4f}"))

# 4. 함수 결과와 비교
res = stats.f_oneway(*groups.values())
print(f"\n직접 계산한 F = {f_manual:.6f}")
print(f"f_oneway  F = {res.statistic:.6f}")
print(f"p-value     = {res.pvalue:.6f}")
print("→ 두 값 일치" if abs(f_manual - res.statistic) < 1e-8 else "→ 두 값 불일치")

# 5. 유의할 때만 사후검정 실행
if res.pvalue < ALPHA:
    tukey = pairwise_tukeyhsd(anova_df["SalePrice"], anova_df["group"], alpha=ALPHA)
    print("\n[Tukey HSD 사후검정]")
    print(tukey)

    tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])

    # 6. reject=True 인 쌍 확인
    sig = tukey_df[tukey_df["reject"] == True]
    print("\n[유의한 집단 쌍]")
    print(sig[["group1", "group2", "meandiff", "p-adj", "reject"]] if len(sig)
          else "유의한 쌍 없음")

    # 7. 집단 쌍별 평균 차이 직접 계산
    print("\n[집단 쌍별 평균 차이]")
    diffs = {}
    for a, b in combinations(sorted(groups), 2):
        d = groups[b].mean() - groups[a].mean()
        diffs[(a, b)] = d
        print(f"{a} vs {b} : {d:,.2f}")

    # 8. 해석
    print("\n[해석]")
    if len(sig):
        pairs = ", ".join(f"{r.group1}-{r.group2}" for r in sig.itertuples())
        print(f"유의하게 차이 나는 쌍 : {pairs}")
        biggest = max(diffs, key=lambda k: abs(diffs[k]))
        print(f"평균 차이가 가장 큰 쌍 : {biggest[0]}-{biggest[1]} "
              f"({abs(diffs[biggest]):,.2f}달러)")
        print("→ 전반적 품질 등급이 높을수록 평균 판매가격이 높아지는 경향으로 해석된다.")
    else:
        print("ANOVA는 유의했으나 Tukey에서는 유의한 쌍이 나오지 않았다.")
else:
    print(f"\n→ ANOVA p ≥ {ALPHA} : 사후검정을 실행하지 않는다.")

[anova_df]
   SalePrice  group
0     132500      5
1     120000      5
2     110000      5
3     172500      5
4     144500      5
전체 행 수 : 60

전체 평균 = 172,008.40

[집단별 평균]
       count      mean
group                 
5         20  130605.0
6         20  167826.6
7         20  217593.6

[ANOVA 표]
                  제곱합(SS)  자유도(df)            평균제곱(MS)       F
집단 간  76,194,788,836.8000        2 38,097,394,418.4000 24.2456
집단 내  89,564,856,787.6000       57  1,571,313,276.9754     NaN
전체   165,759,645,624.4000       59                 NaN     NaN

직접 계산한 F = 24.245575
f_oneway  F = 24.245575
p-value     = 0.000000
→ 두 값 일치

[Tukey HSD 사후검정]
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414 

### 필수 2 답변 작성란

- **Q1.**
-> SS_between은 집단 평균들이 전체 평균에서 벗어난 집단 간 변동 값
- **Q2.**
-> 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미
- **Q3.**
-> 5-6, 5-7, 6-7의 모든 집단 쌍에서 유의한 차이가 확인된다
- **Q4.**
-> 품질 5점과 7점 집단이며 평균 차이는 약 86,988달러이다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [1]:
# 과제 코드를 작성하세요.

import itertools

import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv("ames_housing.csv")

GROUPS = ["Ex", "Gd", "TA"]
N = 20
SEED = 18
ALPHA = 0.05

# 1. 집단별 표본 추출
samples = {
    g: df.loc[df["KitchenQual"] == g, "SalePrice"].sample(n=N, random_state=SEED)
    for g in GROUPS
}

# 2. 표본 수와 평균
print("2) 집단별 표본 수와 평균")
for g in GROUPS:
    print(f"   {g}: n = {len(samples[g])}, 평균 = {samples[g].mean():,.2f}")
print()

# 3. 정규성(Shapiro-Wilk) + 등분산성(Levene)
print("3) 가정 확인")
print("   [정규성 - Shapiro-Wilk]")
normal_ok = True
for g in GROUPS:
    w, p = stats.shapiro(samples[g])
    normal_ok &= p > ALPHA
    print(f"   {g}: W = {w:.4f}, p = {p:.4f} -> {'충족' if p > ALPHA else '위배'}")

lev_stat, lev_p = stats.levene(*[samples[g] for g in GROUPS])
var_ok = lev_p > ALPHA
print("   [등분산성 - Levene]")
print(f"   통계량 = {lev_stat:.4f}, p = {lev_p:.4f} -> {'충족' if var_ok else '위배'}")
if not (normal_ok and var_ok):
    print("   * 가정이 깨졌다면 Kruskal-Wallis 또는 Welch ANOVA를 고려하세요.")
print()

# 4. 일원배치 ANOVA
f_stat, p_value = stats.f_oneway(*[samples[g] for g in GROUPS])
print("4) 일원배치 ANOVA")
print(f"   F통계량 = {f_stat:.4f}")
print(f"   p-value = {p_value:.10f}")
print(f"   -> {'유의함' if p_value < ALPHA else '유의하지 않음'} (유의수준 {ALPHA})\n")

# 5~6. ANOVA가 유의할 때만 Tukey HSD
if p_value < ALPHA:
    long_df = pd.DataFrame({
        "SalePrice": pd.concat([samples[g] for g in GROUPS], ignore_index=True),
        "KitchenQual": sum([[g] * N for g in GROUPS], []),
    })

    tukey = pairwise_tukeyhsd(
        endog=long_df["SalePrice"],
        groups=long_df["KitchenQual"],
        alpha=ALPHA,
    )
    print("5) Tukey HSD 사후 검정")
    print(tukey, "\n")

    result = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
    sig_pairs = result[result["reject"] == True]

    print("6) 유의한 집단 쌍")
    if sig_pairs.empty:
        print("   유의한 쌍이 없습니다.")
    else:
        for _, row in sig_pairs.iterrows():
            print(f"   {row['group1']} vs {row['group2']}: p-adj = {float(row['p-adj']):.4f}")
    print()
else:
    print("5~6) ANOVA가 유의하지 않아 사후 검정을 생략합니다.\n")

# 7. 집단 쌍별 평균 차이
print("7) 집단 쌍별 평균 판매가격 차이")
diffs = {}
for g1, g2 in itertools.combinations(GROUPS, 2):
    d = samples[g1].mean() - samples[g2].mean()
    diffs[(g1, g2)] = d
    print(f"   {g1} - {g2}: {d:,.2f}")
print()

# 8. 해석
top_pair, top_diff = max(diffs.items(), key=lambda kv: abs(kv[1]))
order = sorted(GROUPS, key=lambda g: samples[g].mean(), reverse=True)

print("8) 해석")
print(f"   평균 순위: {' > '.join(order)}")
print(f"   차이가 가장 큰 쌍: {top_pair[0]} vs {top_pair[1]} (차이 {abs(top_diff):,.2f})")

2) 집단별 표본 수와 평균
   Ex: n = 20, 평균 = 313,983.05
   Gd: n = 20, 평균 = 188,835.00
   TA: n = 20, 평균 = 137,486.60

3) 가정 확인
   [정규성 - Shapiro-Wilk]
   Ex: W = 0.9669, p = 0.6875 -> 충족
   Gd: W = 0.9723, p = 0.8024 -> 충족
   TA: W = 0.9519, p = 0.3973 -> 충족
   [등분산성 - Levene]
   통계량 = 2.5550, p = 0.0866 -> 충족

4) 일원배치 ANOVA
   F통계량 = 54.8000
   p-value = 0.0000000000
   -> 유의함 (유의수준 0.05)

5) Tukey HSD 사후 검정
      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2  meandiff  p-adj     lower        upper     reject
----------------------------------------------------------------
    Ex     Gd -125148.05    0.0 -166883.2108  -83412.8892   True
    Ex     TA -176496.45    0.0 -218231.6108 -134761.2892   True
    Gd     TA   -51348.4 0.0122  -93083.5608   -9613.2392   True
---------------------------------------------------------------- 

6) 유의한 집단 쌍
   Ex vs Gd: p-adj = 0.0000
   Ex vs TA: p-adj = 0.0000
   Gd vs TA: p-adj = 0.0122

7) 집단 쌍별 평균 판매가격 차이
   Ex - Gd: 125,148.0

### 과제 답변 작성란

-**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요? 
-> 유의하다. p-value 값이 0.05보다 작고 Levene등분산성도 p = 0.0866으로 0.05보다 크다. 그러므로 귀무가설을 기각한다.

-**Q2.** 사후 검정은 어떤 조건에서 수행하나요?
-> ANOVA 결과가 유의할때, 그리고 비교할 집단이 3개 이상일 때 수행한다.

-**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?
->  세 쌍 모두 유의하다.

-**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
-> Ex vs TA 이고, 차이는 176,496.45이다.

---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
-> 각 검정마다 위양성 가능성이 있기 때문에 비교 횟수가 늘어날수록 전체 분석에서 한 번 이상 잘못 기각할 확률이 누적이다.

2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
-> 귀무가설은 모든 집단의 모집단과 평균이 같다, 대립가설은 적어도 한 집단의 평균이 다르다라는 것

3. F통계량이 크다는 것은 무엇을 의미하나요?
-> 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미

4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
-> ANOVA는 적어도 한 집단이 다르다는 사실만 알려주며 구체적인 집단 쌍은 알려주지 않음 

5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
-> 비교한 집단 쌍, 평균 차이, 조정돤 P-Value, 신뢰구간, reject 여부